# Fermented Dairy Intake and Cardiometabolic Health: Dietary Data Preparation
 
**Date:** March 2026  
**Dataset:** NHANES 2017–2018

---

## Abstract

This notebook prepares the dietary exposure data for an analysis of the association between fermented dairy consumption and cardiometabolic health outcomes in US adults, using the National Health and Nutrition Examination Survey (NHANES) 2017–2018 cycle. Two 24-hour dietary recall interviews are processed to classify each food item as dairy or fermented dairy, aggregate intake per participant, and average across the two recall days. The output is a single dataset — `data/processed/nhanes_dietary_avg.csv` — containing each participant's average daily nutrient intake alongside their average dairy and fermented dairy consumption, ready for use in regression analyses.


## Setup

We begin by importing the libraries we need and defining the paths to our data files. Using `pathlib.Path` rather than plain strings means the paths work correctly on both Windows and macOS/Linux — Python handles the directory separator for us.

In [1]:
import pandas as pd    # Pandas library - for data analysis
import openpyxl        # Read XLSX files
from pathlib import Path
import sys
import subprocess

# ── Detect environment and locate the project root ───────────────────────────
IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "fb3pfb_nhanes"   # adjust if your repo is named differently

if IN_COLAB:
    repo_path = Path("/content") / REPO_NAME
    if not repo_path.exists():
        print("Cloning repository...")
        subprocess.run(
            ["git", "clone", f"https://github.com/ggkuhnle/{REPO_NAME}.git",
             str(repo_path)],
            check=True
        )
    ROOT = repo_path
else:
    # Local: notebook lives in notebooks/, root is one level up
    ROOT = Path.cwd().parent

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Environment:             ", "Colab" if IN_COLAB else "local")
print("Root:                    ", ROOT)
print("Raw data directory:      ", RAW)
print("Processed data directory:", PROCESSED)


Environment:              local
Root:                     /Users/gunter/Documents/fb3pfb_nhanes
Raw data directory:       /Users/gunter/Documents/fb3pfb_nhanes/data/raw
Processed data directory: /Users/gunter/Documents/fb3pfb_nhanes/data/processed


---

## Data Download

The four NHANES 2017–2018 dietary files are downloaded directly from the CDC public server 
if they are not already present in `data/raw/`. This means the notebook is fully self-contained — 
no manual file transfer is needed for these files.

| File | Contents |
|------|----------|
| `DR1IFF_J.xpt` | Day 1 individual foods |
| `DR2IFF_J.xpt` | Day 2 individual foods |
| `DR1TOT_J.xpt` | Day 1 total nutrient intakes |
| `DR2TOT_J.xpt` | Day 2 total nutrient intakes |


In [2]:
import urllib.request

NHANES_BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles"

NHANES_FILES = [
    "DR1IFF_J.xpt",
    "DR2IFF_J.xpt",
    "DR1TOT_J.xpt",
    "DR2TOT_J.xpt",
]

for fname in NHANES_FILES:
    dest = RAW / fname
    if dest.exists():
        print(f"  already present — skipping: {fname}")
    else:
        url = f"{NHANES_BASE}/{fname}"
        print(f"  downloading {fname} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, dest)
        size_mb = dest.stat().st_size / 1_048_576
        print(f"done ({size_mb:.1f} MB)")

print("\nAll files ready.")


  already present — skipping: DR1IFF_J.xpt
  already present — skipping: DR2IFF_J.xpt
  already present — skipping: DR1TOT_J.xpt
  already present — skipping: DR2TOT_J.xpt

All files ready.


---

## Step 1 — Load the food classification lookup table

The USDA maintains a **Food and Nutrient Database for Dietary Studies (FNDDS)**, which assigns a unique integer *food code* to every food and beverage. Each NHANES dietary recall record references these codes, so we can link what a participant ate to a standardised food description.

For this analysis, we use an extended version of the standard FNDDS "At A Glance" spreadsheet with two binary classification columns:

| Column | Meaning |
|--------|--------|
| `Dairy binary` | 1 if the food is a dairy product, 0 otherwise |
| `Fermented dairy binary` | 1 if the food is a *fermented* dairy product (e.g., yogurt, cheese, kefir), 0 otherwise |

The spreadsheet has a long title in row 1 and column headers in row 2; actual data begins in row 3. We skip the title row and read only the three columns we need: the food code (column index 0), the dairy flag (index 7), and the fermented dairy flag (index 8).

We use `openpyxl` with `data_only=True` because the file contains Excel formulas whose cached (pre-calculated) values we want — without this flag, openpyxl would return the formula text rather than the numeric result.

In [5]:
fndds_path = PROCESSED / "2017-2018 FNDDS At A Glance - Foods and Beverages.xlsx"

wb = openpyxl.load_workbook(fndds_path, data_only=True)
ws = wb["Food and Beverages"]

# The FNDDS spreadsheet has a long title in row 1 and column headers in row 2.
# Actual data begins in row 3, so we tell openpyxl to start there (min_row=3).
# values_only=True means each `row` comes back as a plain tuple of cell values
# rather than openpyxl Cell objects — simpler to work with.

# `rows` is a plain Python list that we will fill one dictionary at a time,
# one dictionary per food item in the spreadsheet.
# Each dictionary has three keys: the food code and the two binary flags.
# We build the list this way (rather than reading the whole sheet at once)
# because we only need 3 of the ~80 columns in the spreadsheet.
rows = []

for row in ws.iter_rows(min_row=3, values_only=True):
    # `row` is a tuple representing one spreadsheet row, e.g.:
    #   (11000000, 'Milk, cow's, fluid', ..., 1, 0, ...)
    # We pick out only the three columns we need by their index (0-based).

    food_code  = row[0]   # col A: USDA integer food code, e.g. 11000000
    is_dairy   = row[7]   # col H: 1 if this food is any dairy product, else 0
    is_fermented = row[8] # col I: 1 if this food is a fermented dairy product
                          #        (yogurt, cheese, kefir, etc.), else 0

    # The spreadsheet has a few empty rows at the bottom — once we hit one
    # (food_code is None) we have run out of real data, so we skip it.
    if food_code is not None:
        rows.append({
            "food_code":          int(food_code),

            # is_dairy and is_fermented should always be 0 or 1, but a small
            # number of cells are blank (None) in the spreadsheet.
            # We treat a blank flag as 0 (= not that category) rather than
            # letting it propagate as NaN later.
            "is_dairy":           int(is_dairy)     if is_dairy     is not None else 0,
            "is_fermented_dairy": int(is_fermented) if is_fermented is not None else 0,
        })

# `food_lookup` is a pandas DataFrame — essentially a table — built from the
# list of dictionaries above. pandas automatically uses the dictionary keys
# ("food_code", "is_dairy", "is_fermented_dairy") as column names, and each
# dictionary becomes one row. The result looks like:
#
#    food_code  is_dairy  is_fermented_dairy
#     11000000         1                   0   ← plain milk
#     11112120         1                   1   ← yogurt
#     24198739         0                   0   ← not a dairy food
#
# This table is used in Step 3 to label every food item a participant ate
# as dairy / fermented dairy / neither, via a merge on food_code.
food_lookup = pd.DataFrame(rows)

---

## Step 2 — Load the NHANES individual foods files

NHANES collects dietary information through **24-hour dietary recalls** — participants are interviewed by a trained nutritionist and asked to describe everything they ate and drank in the previous 24 hours. In the 2017–2018 cycle, most participants completed two such recalls: one in person (Day 1) and a follow-up by phone (Day 2).

The **individual foods files** (`DR1IFF_J` for Day 1, `DR2IFF_J` for Day 2) contain one row for each food item a participant reported — so a single participant will have as many rows as foods they described. The key variables are:

| Variable | Description |
|----------|------------|
| `SEQN` | Unique participant identifier |
| `DR1IFDCD` / `DR2IFDCD` | USDA food code (links to the FNDDS lookup) |
| `DR1IGRMS` / `DR2IGRMS` | Gram weight of the food item consumed |

The files are stored in SAS XPORT format (`.xpt`), which `pandas` can read directly.

In [6]:
# Load Day 1 individual foods
iff_day1_raw = pd.read_sas(RAW / "DR1IFF_J.xpt", format="xport", encoding="utf-8")
print("Day 1 individual foods — shape:", iff_day1_raw.shape)
print("Columns:", list(iff_day1_raw.columns))
iff_day1_raw[["SEQN", "DR1IFDCD", "DR1IGRMS"]].head()

Day 1 individual foods — shape: (112683, 84)
Columns: ['SEQN', 'WTDRD1', 'WTDR2D', 'DR1ILINE', 'DR1DRSTZ', 'DR1EXMER', 'DRABF', 'DRDINT', 'DR1DBIH', 'DR1DAY', 'DR1LANG', 'DR1CCMNM', 'DR1CCMTX', 'DR1_020', 'DR1_030Z', 'DR1FS', 'DR1_040Z', 'DR1IFDCD', 'DR1IGRMS', 'DR1IKCAL', 'DR1IPROT', 'DR1ICARB', 'DR1ISUGR', 'DR1IFIBE', 'DR1ITFAT', 'DR1ISFAT', 'DR1IMFAT', 'DR1IPFAT', 'DR1ICHOL', 'DR1IATOC', 'DR1IATOA', 'DR1IRET', 'DR1IVARA', 'DR1IACAR', 'DR1IBCAR', 'DR1ICRYP', 'DR1ILYCO', 'DR1ILZ', 'DR1IVB1', 'DR1IVB2', 'DR1INIAC', 'DR1IVB6', 'DR1IFOLA', 'DR1IFA', 'DR1IFF', 'DR1IFDFE', 'DR1ICHL', 'DR1IVB12', 'DR1IB12A', 'DR1IVC', 'DR1IVD', 'DR1IVK', 'DR1ICALC', 'DR1IPHOS', 'DR1IMAGN', 'DR1IIRON', 'DR1IZINC', 'DR1ICOPP', 'DR1ISODI', 'DR1IPOTA', 'DR1ISELE', 'DR1ICAFF', 'DR1ITHEO', 'DR1IALCO', 'DR1IMOIS', 'DR1IS040', 'DR1IS060', 'DR1IS080', 'DR1IS100', 'DR1IS120', 'DR1IS140', 'DR1IS160', 'DR1IS180', 'DR1IM161', 'DR1IM181', 'DR1IM201', 'DR1IM221', 'DR1IP182', 'DR1IP183', 'DR1IP184', 'DR1IP204', 'DR1IP205',

,SEQN,DR1IFDCD,DR1IGRMS
0,93704.0,55100050.0,80.58
1,93704.0,11513600.0,248.00
2,93704.0,24198739.0,100.00
3,93704.0,74401010.0,5.67
4,93704.0,71401020.0,112.00


In [7]:
# Load Day 2 individual foods
iff_day2_raw = pd.read_sas(RAW / "DR2IFF_J.xpt", format="xport", encoding="utf-8")
print("Day 2 individual foods — shape:", iff_day2_raw.shape)
print("Columns:", list(iff_day2_raw.columns))
iff_day2_raw[["SEQN", "DR2IFDCD", "DR2IGRMS"]].head()

Day 2 individual foods — shape: (93500, 84)
Columns: ['SEQN', 'WTDRD1', 'WTDR2D', 'DR2ILINE', 'DR2DRSTZ', 'DR2EXMER', 'DRABF', 'DRDINT', 'DR2DBIH', 'DR2DAY', 'DR2LANG', 'DR2CCMNM', 'DR2CCMTX', 'DR2_020', 'DR2_030Z', 'DR2FS', 'DR2_040Z', 'DR2IFDCD', 'DR2IGRMS', 'DR2IKCAL', 'DR2IPROT', 'DR2ICARB', 'DR2ISUGR', 'DR2IFIBE', 'DR2ITFAT', 'DR2ISFAT', 'DR2IMFAT', 'DR2IPFAT', 'DR2ICHOL', 'DR2IATOC', 'DR2IATOA', 'DR2IRET', 'DR2IVARA', 'DR2IACAR', 'DR2IBCAR', 'DR2ICRYP', 'DR2ILYCO', 'DR2ILZ', 'DR2IVB1', 'DR2IVB2', 'DR2INIAC', 'DR2IVB6', 'DR2IFOLA', 'DR2IFA', 'DR2IFF', 'DR2IFDFE', 'DR2ICHL', 'DR2IVB12', 'DR2IB12A', 'DR2IVC', 'DR2IVD', 'DR2IVK', 'DR2ICALC', 'DR2IPHOS', 'DR2IMAGN', 'DR2IIRON', 'DR2IZINC', 'DR2ICOPP', 'DR2ISODI', 'DR2IPOTA', 'DR2ISELE', 'DR2ICAFF', 'DR2ITHEO', 'DR2IALCO', 'DR2IMOIS', 'DR2IS040', 'DR2IS060', 'DR2IS080', 'DR2IS100', 'DR2IS120', 'DR2IS140', 'DR2IS160', 'DR2IS180', 'DR2IM161', 'DR2IM181', 'DR2IM201', 'DR2IM221', 'DR2IP182', 'DR2IP183', 'DR2IP184', 'DR2IP204', 'DR2IP205', 

,SEQN,DR2IFDCD,DR2IGRMS
0,93704.0,64104010.0,155.00
1,93704.0,57241000.0,27.75
2,93704.0,11112210.0,122.00
3,93704.0,63149010.0,77.50
4,93704.0,52302010.0,130.00


In [8]:
# Keep only the variables we need and standardise types
iff_day1 = iff_day1_raw[["SEQN", "DR1IFDCD", "DR1IGRMS"]].copy()
iff_day2 = iff_day2_raw[["SEQN", "DR2IFDCD", "DR2IGRMS"]].copy()

# SEQN and food codes come through as float64 from the XPT reader; convert to int
iff_day1["SEQN"] = iff_day1["SEQN"].astype(int)
iff_day1["DR1IFDCD"] = iff_day1["DR1IFDCD"].astype(int)

iff_day2["SEQN"] = iff_day2["SEQN"].astype(int)
iff_day2["DR2IFDCD"] = iff_day2["DR2IFDCD"].astype(int)

print("Day 1 unique participants:", iff_day1["SEQN"].nunique())
print("Day 2 unique participants:", iff_day2["SEQN"].nunique())

Day 1 unique participants: 7640
Day 2 unique participants: 6639


---

## Step 3 — Classify each food item as dairy / fermented dairy

Each row in the individual foods file contains a food code, but not yet a dairy flag. We need to bring in that information from the lookup table we built in Step 1. This is done with a **merge** (also called a join): we match rows from two tables based on a shared key column — in this case the food code.

We use a **left merge**, meaning we keep every row from the left-hand table (the individual foods file) regardless of whether a matching food code exists in the right-hand table (the lookup). This matters because:

- Not every food code in the dietary recall may appear in the FNDDS file (e.g. codes for combination foods or commercial products may differ slightly).
- We do not want to silently drop food records — we want to retain them and simply treat unmatched codes as non-dairy (flag = 0).

After merging, any row that did not find a match will have `NaN` in the flag columns; we fill those with 0.

In [9]:
# --- Day 1 ---
iff_day1_classified = iff_day1.merge(
    food_lookup,
    left_on="DR1IFDCD",   # food code column in the Day 1 file
    right_on="food_code", # food code column in the lookup
    how="left",           # keep all Day 1 rows
)

# Fill NaN flags (unmatched food codes) with 0
iff_day1_classified["is_dairy"] = iff_day1_classified["is_dairy"].fillna(0).astype(int)
iff_day1_classified["is_fermented_dairy"] = iff_day1_classified["is_fermented_dairy"].fillna(0).astype(int)

# A left merge must not change the row count — assert this as a sanity check
assert len(iff_day1_classified) == len(iff_day1), (
    f"Row count changed after merge! Expected {len(iff_day1)}, got {len(iff_day1_classified)}. "
    "Check for duplicate food codes in the lookup table."
)

unmatched_day1 = iff_day1_classified["food_code"].isna().sum()
print(f"Day 1: {len(iff_day1_classified)} food records | {unmatched_day1} unmatched food codes (treated as non-dairy)")
print(f"Dairy records: {iff_day1_classified['is_dairy'].sum()} | Fermented dairy records: {iff_day1_classified['is_fermented_dairy'].sum()}")
iff_day1_classified.head()

Day 1: 112683 food records | 0 unmatched food codes (treated as non-dairy)
Dairy records: 19573 | Fermented dairy records: 8981


,SEQN,DR1IFDCD,DR1IGRMS,food_code,is_dairy,is_fermented_dairy
0,93704,55100050,80.58,55100050,1,1
1,93704,11513600,248.00,11513600,1,0
2,93704,24198739,100.00,24198739,0,0
3,93704,74401010,5.67,74401010,0,0
4,93704,71401020,112.00,71401020,0,0


In [10]:
# --- Day 2 ---
iff_day2_classified = iff_day2.merge(
    food_lookup,
    left_on="DR2IFDCD",
    right_on="food_code",
    how="left",
)

iff_day2_classified["is_dairy"] = iff_day2_classified["is_dairy"].fillna(0).astype(int)
iff_day2_classified["is_fermented_dairy"] = iff_day2_classified["is_fermented_dairy"].fillna(0).astype(int)

assert len(iff_day2_classified) == len(iff_day2), (
    f"Row count changed after merge! Expected {len(iff_day2)}, got {len(iff_day2_classified)}. "
    "Check for duplicate food codes in the lookup table."
)

unmatched_day2 = iff_day2_classified["food_code"].isna().sum()
print(f"Day 2: {len(iff_day2_classified)} food records | {unmatched_day2} unmatched food codes (treated as non-dairy)")
print(f"Dairy records: {iff_day2_classified['is_dairy'].sum()} | Fermented dairy records: {iff_day2_classified['is_fermented_dairy'].sum()}")
iff_day2_classified.head()

Day 2: 93500 food records | 0 unmatched food codes (treated as non-dairy)
Dairy records: 16107 | Fermented dairy records: 7071


,SEQN,DR2IFDCD,DR2IGRMS,food_code,is_dairy,is_fermented_dairy
0,93704,64104010,155.00,64104010,0,0
1,93704,57241000,27.75,57241000,0,0
2,93704,11112210,122.00,11112210,1,0
3,93704,63149010,77.50,63149010,0,0
4,93704,52302010,130.00,52302010,0,0


---

## Step 4 — Summarise dairy intake per participant per day

Currently we have one row per *food item*. We want one row per *participant* per *day*, showing how many grams of dairy and fermented dairy they consumed in total.

The logic is:
1. Filter to rows where `is_dairy == 1` and sum the gram weights, grouped by participant (`SEQN`). This gives `total_dairy_g`.
2. Repeat for `is_fermented_dairy == 1` to get `fermented_dairy_g`.
3. Combine these two series into a single summary DataFrame.

**Handling zero consumers:** Participants who ate no dairy on a given day will not appear in the filtered subsets at all. We use `.reindex()` to re-insert every participant from the full individual foods file, assigning 0 grams where they are absent. This is important — we must not drop non-dairy consumers from the dataset.

In [ ]:
def summarise_dairy_intake(classified_df, gram_col):
    """
    Compute total dairy and fermented dairy grams per participant.

    Parameters
    ----------
    classified_df : DataFrame with columns SEQN, <gram_col>, is_dairy, is_fermented_dairy
    gram_col      : name of the column holding gram weights (e.g. 'DR1IGRMS')

    Returns
    -------
    DataFrame with columns: SEQN, total_dairy_g, fermented_dairy_g
    """
    # Get the complete list of unique participant IDs from the dataset.
    # We need this upfront so we can re-insert participants who ate no dairy —
    # they would otherwise disappear from the grouped result entirely.

    all_seqns = classified_df["SEQN"].unique()

    # Sum gram weights of dairy foods per participant.
    # The result is a pandas Series (a single column of values, one per participant).
    
    dairy_g = (
        # .loc[] filters rows by a condition — keep only rows where is_dairy == 1,
        # i.e. only the food items that are dairy products.
        # This is equivalent to: classified_df[classified_df["is_dairy"] == 1]
        classified_df.loc[classified_df["is_dairy"] == 1]

        # .groupby("SEQN") splits the filtered rows into groups, one group per
        # participant. Everything after this operates on each group separately.
        .groupby("SEQN")[gram_col]   # [gram_col] selects e.g. "DR1IGRMS" within each group

        # .sum() adds up the gram weights within each group, giving one total per
        # participant — but only for participants who ate at least one dairy item.
        .sum()

        # .reindex() ensures every participant ID in all_seqns appears in the result.
        # Participants who ate no dairy have no group in the groupby output, so they
        # are missing. reindex re-inserts them and assigns fill_value=0 (not NaN).
        .reindex(all_seqns)

        # .rename() gives the Series a name, which becomes the column name when we
        # combine multiple Series into a DataFrame later.
        .rename("total_dairy_g")
    )

    # Exactly the same process, but filtering on is_fermented_dairy == 1 instead.
    fermented_g = (
        classified_df.loc[classified_df["is_fermented_dairy"] == 1]
        .groupby("SEQN")[gram_col]
        .sum()
        .reindex(all_seqns)
        .rename("fermented_dairy_g")
    )

    # pd.concat() combines the two Series side by side into a single DataFrame.
    # axis=1 means "stack them as columns" (axis=0 would stack them as rows).
    # At this point SEQN is the row index (the row label), not a regular column.
    #
    #              total_dairy_g  fermented_dairy_g
    # SEQN
    # 93704               356.93             108.93
    # 93705                71.45              41.45
    #
    summary = pd.concat([dairy_g, fermented_g], axis=1)

    # .reset_index() promotes the row index (SEQN) into a regular column,
    # replacing it with a plain integer index 0, 1, 2, ...
    # This is the pandas equivalent of tibble::rownames_to_column() in R.
    #
    #     SEQN  total_dairy_g  fermented_dairy_g
    # 0  93704         356.93             108.93
    # 1  93705          71.45              41.45
    #
    summary = summary.reset_index()

    # reset_index() names the new column "SEQN" automatically when the index
    # already had that name — but if it didn't, it would be called "index".
    # This rename is a defensive step to make sure the column is always called SEQN.
    summary.rename(columns={"index": "SEQN"}, inplace=True)

    return summary

dairy_day1 = summarise_dairy_intake(iff_day1_classified, "DR1IGRMS")
dairy_day2 = summarise_dairy_intake(iff_day2_classified, "DR2IGRMS")

print("Day 1 dairy summary — shape:", dairy_day1.shape)
print(dairy_day1.describe())
dairy_day1.head()

Day 1 dairy summary — shape: (7640, 3)
                SEQN  total_dairy_g  fermented_dairy_g
count    7640.000000    7640.000000        7640.000000
mean    98324.203534     330.721124         116.296082
std      2669.054027     333.804619         168.750172
min     93704.000000       0.000000           0.000000
25%     96003.750000      70.000000           0.000000
50%     98335.000000     250.000000          42.500000
75%    100624.250000     490.137500         172.085000
max    102956.000000    3924.280000        1690.890000


,SEQN,total_dairy_g,fermented_dairy_g
0,93704,356.93,108.93
1,93705,71.45,41.45
2,93706,250.50,250.50
3,93707,150.00,0.00
4,93708,18.00,2.75


In [12]:
print("Day 2 dairy summary — shape:", dairy_day2.shape)
print(dairy_day2.describe())
dairy_day2.head()

Day 2 dairy summary — shape: (6639, 3)
                SEQN  total_dairy_g  fermented_dairy_g
count    6639.000000    6639.000000        6639.000000
mean    98315.563790     321.959563         109.575674
std      2674.902408     332.595773         169.387928
min     93704.000000       0.000000           0.000000
25%     95968.500000      61.000000           0.000000
50%     98320.000000     244.000000          33.750000
75%    100632.500000     485.655000         163.320000
max    102956.000000    5014.000000        2803.500000


,SEQN,total_dairy_g,fermented_dairy_g
0,93704,515.40,145.4
1,93705,77.00,77.0
2,93707,0.00,0.0
3,93708,350.88,0.0
4,93710,239.10,170.1


---

## Step 5 — Load the total nutrient intake files and average across two days

In addition to the food-level files, NHANES provides **total nutrient intake files** (`DR1TOT_J` and `DR2TOT_J`) that contain one row per participant per day, with pre-calculated totals for energy, macronutrients, vitamins, minerals, and so on. These are the variables we will use as covariates in the analysis.

**Why average across two days rather than using just one?**  
A single 24-hour recall captures what a person happened to eat on one particular day, which may be quite atypical for that individual — this is called *within-person day-to-day variability*. Averaging across two recall days reduces this random noise and gives a better estimate of *habitual* intake. This is standard practice in dietary epidemiology.

In [13]:
# Load the two total nutrient files
tot_day1 = pd.read_sas(RAW / "DR1TOT_J.xpt", format="xport", encoding="utf-8")
tot_day2 = pd.read_sas(RAW / "DR2TOT_J.xpt", format="xport", encoding="utf-8")

print("DR1TOT shape:", tot_day1.shape, "| Columns:", list(tot_day1.columns[:10]), "...")
print("DR2TOT shape:", tot_day2.shape)
tot_day1.head(3)

DR1TOT shape: (8704, 168) | Columns: ['SEQN', 'WTDRD1', 'WTDR2D', 'DR1DRSTZ', 'DR1EXMER', 'DRABF', 'DRDINT', 'DR1DBIH', 'DR1DAY', 'DR1LANG'] ...
DR2TOT shape: (8704, 85)


,SEQN,WTDRD1,WTDR2D,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,DR1LANG,...,DRD370QQ,DRD370R,DRD370RQ,DRD370S,DRD370SQ,DRD370T,DRD370TQ,DRD370U,DRD370UQ,DRD370V
0,93703.0,5.397605e-79,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,8.171401e+04,82442.869214,1.0,49.0,2.0,2.0,7.0,2.0,1.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0
2,93705.0,7.185561e+03,5640.391078,1.0,73.0,2.0,2.0,5.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ── Combine the two total nutrient files into one long table ─────────────────
#
# tot_day1 and tot_day2 each have one row per participant.
# We stack them vertically (axis=0) to get a table with two rows per
# participant — one for each recall day. This "long" format makes it easy
# to average across days using groupby in the next step.
#
# Before stacking:
#   tot_day1:  8704 rows × 168 cols   (one row per participant, Day 1)
#   tot_day2:  8704 rows ×  85 cols   (one row per participant, Day 2)
#
# After stacking:
#   tot_both_days:  17408 rows × 248 cols  (two rows per participant)
#
# ignore_index=True resets the row numbers to 0, 1, 2, ... after stacking.
# Without it, the row numbers from tot_day1 (0–8703) and tot_day2 (0–8703)
# would both appear, giving duplicate row numbers which can cause confusion.

tot_both_days = pd.concat([tot_day1, tot_day2], axis=0, ignore_index=True)
print("Combined (long) shape:", tot_both_days.shape)

# The XPT reader returns SEQN as float64 (e.g. 93704.0).
# We convert to int so it matches the SEQN column in our other DataFrames
# and doesn't cause a mismatch when merging later.

tot_both_days["SEQN"] = tot_both_days["SEQN"].astype(int)

# ── Average nutrient values across the two recall days ───────────────────────
#
# .groupby("SEQN") splits the long table into groups, one group per participant.
# Each group has two rows — one for Day 1 and one for Day 2.
#
# .mean() then calculates the column-wise average across those two rows,
# giving a single row per participant representing their average daily intake.
#
# numeric_only=True tells pandas to skip any text columns that cannot be
# averaged — without it, pandas would raise an error on non-numeric columns.
#
# .reset_index() promotes SEQN from the row index back into a regular column,
# as we saw in the summarise_dairy_intake function earlier.
#
# The result has the same number of rows as participants (8704), but now each
# row is the mean of Day 1 and Day 2 rather than a single day's values.

nutrients_avg = tot_both_days.groupby("SEQN").mean(numeric_only=True).reset_index()

print("Averaged dataset shape:", nutrients_avg.shape)

# .nunique() counts the number of distinct values in the SEQN column.
# This is a sanity check: if the number of unique SEQNs equals the number
# of rows, we know groupby produced exactly one row per participant as expected.
print(f"Unique participants: {nutrients_avg['SEQN'].nunique()}")
nutrients_avg.head(3)


Combined (long) shape: (17408, 248)
Averaged dataset shape: (8704, 248)
Unique participants: 8704


,SEQN,WTDRD1,WTDR2D,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,DR1LANG,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWSZ
0,93703,5.397605e-79,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704,8.171401e+04,82442.869214,1.0,49.0,2.0,2.0,7.0,2.0,1.0,...,0.003,0.041,5.397605e-79,0.002,0.010,2.0,5.397605e-79,5.397605e-79,5.397605e-79,91.0
2,93705,7.185561e+03,5640.391078,1.0,73.0,2.0,2.0,5.0,1.0,1.0,...,0.001,0.201,2.000000e-03,0.011,0.059,2.0,9.600000e+02,5.397605e-79,9.600000e+02,1.0


---

## Step 6 — Add dairy variables to the averaged nutrient dataset

We now have two pieces of information for each participant:

1. **`nutrients_avg`** — average daily nutrient intakes across two recall days.
2. **`dairy_day1` / `dairy_day2`** — dairy and fermented dairy grams for each individual recall day.

We calculate each participant's average dairy intake across the two days, then merge that into `nutrients_avg`. Participants who appear in the nutrient totals file but have no individual food records (e.g. due to data quality exclusions) are assigned 0 grams of dairy.

In [15]:
# ── Combine the two per-day dairy summaries into one long table ──────────────
#
# dairy_day1 and dairy_day2 each have one row per participant, holding their
# total dairy and fermented dairy grams for that recall day.
#
# .assign(recall_day=1) and .assign(recall_day=2) add a new column to each
# DataFrame before stacking, labelling which day each row came from.
# This is not strictly needed for the averaging below, but it makes the
# intermediate table easier to inspect and debug if something goes wrong.
#
# axis=0 stacks rows vertically (same pattern as with the nutrient totals).
# ignore_index=True resets row numbers to avoid duplicate indices.
dairy_both_days = pd.concat(
    [dairy_day1.assign(recall_day=1), dairy_day2.assign(recall_day=2)],
    axis=0,
    ignore_index=True,
)

# ── Average dairy intake across the two recall days ──────────────────────────
dairy_avg = (
    dairy_both_days

    # Group by participant — each group has one row (Day 1 only) or two rows
    # (Day 1 and Day 2). We select only the two columns we want to average;
    # recall_day is intentionally excluded so it doesn't get averaged too.
    .groupby("SEQN")[["total_dairy_g", "fermented_dairy_g"]]

    # Average across the rows in each group.
    # Participants with only one recall day get that day's value as-is —
    # NaN is ignored rather than treated as 0, so their intake is not
    # artificially halved.
    .mean()

    # Promotes SEQN from the row index back into a regular column.
    .reset_index()

    # Rename to make clear these are averages across days, not single-day values.
    .rename(columns={
        "total_dairy_g": "avg_total_dairy_g",
        "fermented_dairy_g": "avg_fermented_dairy_g",
    })
)

# Shape should be one row per participant who completed at least one recall.
print("Dairy averages shape:", dairy_avg.shape)

# .describe() gives a quick statistical summary (mean, sd, min, quartiles, max)
# for each column — useful for a sanity check before merging into the main dataset.
print(dairy_avg.describe())
dairy_avg.head()

Dairy averages shape: (7641, 3)
                SEQN  avg_total_dairy_g  avg_fermented_dairy_g
count    7641.000000        7641.000000            7641.000000
mean    98323.610391         327.147794             113.028929
std      2669.382925         287.669847             136.667934
min     93704.000000           0.000000               0.000000
25%     96003.000000         119.000000              10.500000
50%     98334.000000         265.250000              71.000000
75%    100624.000000         463.215000             166.875000
max    102956.000000        3308.860000            2209.375000


,SEQN,avg_total_dairy_g,avg_fermented_dairy_g
0,93704,436.165,127.165
1,93705,74.225,59.225
2,93706,250.500,250.500
3,93707,75.000,0.000
4,93708,184.440,1.375


In [16]:
# Merge the dairy averages into the nutrient averages
# left merge: keep all participants from nutrients_avg
# Participants absent from dairy_avg (no food-level records) get NaN → fill with 0
nutrients_with_dairy = nutrients_avg.merge(dairy_avg, on="SEQN", how="left")

nutrients_with_dairy["avg_total_dairy_g"] = nutrients_with_dairy["avg_total_dairy_g"].fillna(0)
nutrients_with_dairy["avg_fermented_dairy_g"] = nutrients_with_dairy["avg_fermented_dairy_g"].fillna(0)

# Sanity check: row count must not change
assert len(nutrients_with_dairy) == len(nutrients_avg), (
    f"Unexpected row count after merge: {len(nutrients_with_dairy)} vs {len(nutrients_avg)}. "
    "The dairy_avg table may contain duplicate SEQNs."
)

print("Final shape:", nutrients_with_dairy.shape)
print("\nDairy variable summary:")
print(nutrients_with_dairy[["avg_total_dairy_g", "avg_fermented_dairy_g"]].describe())
nutrients_with_dairy.head()

Final shape: (8704, 250)

Dairy variable summary:
       avg_total_dairy_g  avg_fermented_dairy_g
count        8704.000000            8704.000000
mean          287.193968              99.224959
std           290.038003             133.291411
min             0.000000               0.000000
25%            50.000000               0.000000
50%           222.642500              52.900000
75%           427.007500             150.000000
max          3308.860000            2209.375000


,SEQN,WTDRD1,WTDR2D,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,DR1LANG,...,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWSZ,avg_total_dairy_g,avg_fermented_dairy_g
0,93703,5.397605e-79,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000,0.000
1,93704,8.171401e+04,8.244287e+04,1.0,49.0,2.0,2.0,7.0,2.0,1.0,...,5.397605e-79,0.002,0.010,2.0,5.397605e-79,5.397605e-79,5.397605e-79,91.0,436.165,127.165
2,93705,7.185561e+03,5.640391e+03,1.0,73.0,2.0,2.0,5.0,1.0,1.0,...,2.000000e-03,0.011,0.059,2.0,9.600000e+02,5.397605e-79,9.600000e+02,1.0,74.225,59.225
3,93706,6.463883e+03,5.397605e-79,1.0,86.0,2.0,1.0,NaN,6.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,250.500,250.500
4,93707,1.533378e+04,2.270707e+04,1.0,81.0,2.0,2.0,14.0,2.0,1.0,...,4.000000e-03,0.025,0.061,3.0,1.020000e+03,7.800000e+02,2.400000e+02,99.0,75.000,0.000


---

## Step 7 — Save the final dataset

The prepared dataset is saved as a CSV file to `data/processed/nhanes_dietary_avg.csv`. CSV (comma-separated values) is a plain-text format that can be opened in Excel, R, or any other analysis tool without needing special software.

We set `index=False` so that pandas does not write its internal row numbers as an extra column — the data already has a proper participant identifier (`SEQN`).

**What this file contains:** One row per NHANES 2017–2018 participant who completed at least one dietary recall. Each row holds their average daily nutrient intakes (averaged across the two recall days), plus two new exposure variables: `avg_total_dairy_g` (average grams of all dairy consumed per day) and `avg_fermented_dairy_g` (average grams of fermented dairy consumed per day).

**How to use it:** This file should be the starting point for subsequent notebooks that merge in health outcome data (e.g. blood pressure, cholesterol, BMI from other NHANES files) and run regression models.

In [ ]:
output_path = PROCESSED / "nhanes_dietary_avg.csv"
nutrients_with_dairy.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {nutrients_with_dairy.shape}")
print("\nFirst 5 rows:")
nutrients_with_dairy.head(5)

---

## Summary

This notebook has produced:

**Output file:** `data/processed/nhanes_dietary_avg.csv`

**Key variables:**

| Variable | Description |
|----------|------------|
| `SEQN` | NHANES participant identifier |
| `avg_total_dairy_g` | Average grams of total dairy consumed per day (mean of Day 1 and Day 2) |
| `avg_fermented_dairy_g` | Average grams of fermented dairy consumed per day (mean of Day 1 and Day 2) |
| All `DR1`/`DR2` nutrient variables | Averaged across two recall days (energy, macronutrients, micronutrients) |

**Processing steps completed:**
1. Food classification lookup table loaded from FNDDS 2017–2018 (Excel).
2. NHANES individual foods files (Days 1 and 2) loaded and merged with the lookup.
3. Dairy and fermented dairy grams aggregated per participant per day.
4. Total nutrient intake files loaded and averaged across two recall days.
5. Dairy exposure variables merged into the nutrient dataset and saved.

**Next step:** Load `nhanes_dietary_avg.csv` in the next notebook alongside the NHANES demographic and health outcome files (e.g. `DEMO_J.xpt`, `BPX_J.xpt`, `TRIGLY_J.xpt`) to build the analytical dataset for regression modelling.